# Istari External Workflow Logs — Interactive Walkthrough

This notebook walks through Istari's **External Workflow Log** feature end-to-end. It's meant to be run cell-by-cell on a call so we can pause and talk through each step.

## The problem it solves

A lot of valuable engineering work happens **outside** Istari — a verification script on an engineer's workstation, a nightly CI job, a solver run on an HPC cluster. That work produces files and a verdict, but the *record* of it usually lives in scattered folders, chat messages, and people's memory. There's no durable link between "this batch of results" and "the exact version of the design it was run against."

**External Workflow Logs** close that gap. From any external process you can:

- upload the files a run produced (**workflow outputs**), and
- record a single **workflow log entry** that ties those files to the **system configuration** that was active when the run happened — with a title and a pass/fail status.

The result is a permanent, queryable history on the system: *what was produced, what the verdict was, and which version of the design it applied to.*

> **Not the same as Istari jobs.** Istari *jobs* run on Istari agents and are scheduled and tracked by the platform. External workflows run wherever you run them — Istari just stores the record and the outputs.

### Prerequisites

From the cookbook root:

```bash
uv sync --group dev
```

Uses **`istari-digital-client`**, **`python-dotenv`**, **`pandas`**, **`jinja2`** (for `DataFrame.style`), **`matplotlib`**, **`numpy`**, **`pytest`**, and **`ipython`** (all in the `dev` dependency group). Local helpers in this folder: `bracket_step.py`, `istari_helpers.py`, `run_workflow.py`, `tradespace_tests.py`.

Set `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN` in [`samples/.env`](../.env). **`istari-digital-client` must match the registry version** — §1 calls `Client.check_compatibility()` and fails fast if they differ. Run with the **Python (istari-client-cookbook)** kernel, or any kernel backed by the synced `.venv`.

## Our scenario: a bracket that must pass a verification battery

We'll use a small, concrete story:

1. A **bracket** (a CAD `.step` file) is tracked on a system in Istari.
2. An engineer runs a local **verification battery** — ten checks (mass, stress, thermal, fatigue, …) that produce result files and an overall pass/fail.
3. The first design **fails** one check (the stress margin is too thin).
4. The engineer **revises the design**, commits the new revision to Istari, and re-runs.
5. The second run **passes**, and the verified test report is written back onto the design.

Every run is captured in the system's **Workflow log**, so the whole design loop is visible in one place.

## 1. Connect to Istari

We read the registry URL and a personal access token from `.env`. These two clients share one configuration:

- `Client` — the v2 client we use to read the system and download the tracked file.
- `V3Client` — the client that exposes workflow outputs and workflow log entries.

The connect cell calls `Client.check_compatibility()` (readiness + response headers) and asserts the installed SDK version matches the registry. It also probes **v3 workflow** API hashes, since those calls happen later in this notebook.

In [ ]:
import os, json, re
from importlib.metadata import version as pkg_version
from pathlib import Path
from dotenv import load_dotenv

from istari_digital_client import Client, V3Client, Configuration
from istari_digital_client.api_client import ApiClient
from istari_digital_client.storage.api.storage_api import StorageApi
from istari_digital_client.v2.api.v2_api import V2Api
from istari_digital_client.v3.api.v3_api import V3Api
from istari_digital_client.v3.models import WorkflowLogEntryCreateDto

load_dotenv("../.env")
REGISTRY_URL = os.environ["ISTARI_REGISTRY_URL"]
PAT = os.environ["ISTARI_PERSONAL_ACCESS_TOKEN"]

_match = re.match(r"^(https?://)(?:fileservice-v2\.)?(.+?)/?$", REGISTRY_URL)
UI_URL = REGISTRY_URL.rstrip("/") if not _match else f"{_match.group(1)}{_match.group(2)}"

config = Configuration(registry_url=REGISTRY_URL, registry_auth_token=PAT)
client = Client(config)     # read system + download files
v3     = V3Client(config)   # workflow outputs + workflow log

installed = pkg_version("istari-digital-client")

compat = client.check_compatibility()
assert compat and compat.server_version, (
    "Registry did not return compatibility headers. "
    "Check ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN in samples/.env."
)
registry_version = compat.server_version

# V3Client has no check_compatibility(); probe workflow API hashes used in §5+.
_v3_probe = ApiClient(config, spec_ids=(V3Api._SPEC_ID, StorageApi._SPEC_ID))
V2Api(config, _v3_probe).readiness_check()
compat_v3 = _v3_probe.compatibility_checker.last_result

assert installed == registry_version, (
    f"istari-digital-client=={installed} does not match Istari Registry "
    f"v{registry_version}. From the cookbook root run "
    f"`uv sync --group dev` (or install istari-digital-client=={registry_version} "
    "when available on PyPI), then restart the notebook kernel."
)
for scope, result in (("v2/storage", compat), ("v3/workflows", compat_v3)):
    assert result and result.status == "compatible", (
        f"SDK is {result.status if result else 'unknown'} with Istari Registry "
        f"v{registry_version} ({scope}; affected APIs: "
        f"{', '.join(result.incompatible_tags) if result else 'none'}). "
        "Update istari-digital-client to match the server before continuing."
    )

print("Connected to:", REGISTRY_URL)
print("UI:", UI_URL)
print(f"istari-digital-client {installed}")
print(f"Istari Registry v{registry_version} — compatible")

## 2. A system to work against

In a real engagement the system and its tracked CAD file already exist in Istari. To keep this notebook self-contained, we create a fresh one here: we upload the **initial bracket** (`bracket.step`) and put it under a **configuration** called `baseline`.

`bracket_step.bad()` generates a synthetic STEP file whose fillet radius is **3 mm** — intentionally too small, as we'll see.

In [ ]:
import bracket_step, istari_helpers
from istari_digital_client.v2.models.new_system import NewSystem
from istari_digital_client.v2.models.new_system_configuration import NewSystemConfiguration
from istari_digital_client.v2.models.new_tracked_file import NewTrackedFile
from istari_digital_client.v2.models.tracked_file_specifier_type import TrackedFileSpecifierType

work = Path("_notebook_run"); work.mkdir(exist_ok=True)
src = work / "bracket.step"
src.write_text(bracket_step.bad())          # initial design: R3 fillet

model  = client.add_model(path=src, display_name="bracket.step",
                          description="Bracket — initial R3 design")
system = client.create_system(NewSystem(name="Workflow Log Walkthrough",
                                        description="External workflow log demo"))
config_obj = client.create_configuration(
    system_id=system.id,
    new_system_configuration=NewSystemConfiguration(
        name="baseline",
        tracked_files=[NewTrackedFile(specifier_type=TrackedFileSpecifierType.LATEST,
                                      file_id=model.file.id)],
    ),
)

SYSTEM_ID, CONFIG_ID, MODEL_ID, FILE_ID = system.id, config_obj.id, model.id, model.file.id

# Make the bracket show up when the system is opened in the UI.
istari_helpers.show_config_on_system(client, SYSTEM_ID, CONFIG_ID)

print("System:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 3. Pull the design out of Istari

The external workflow starts by fetching the file it needs to act on. We download the latest revision of the tracked bracket and read its bytes — exactly what a real script would do before handing the geometry to a solver.

In [ ]:
file_obj = client.get_file(file_id=FILE_ID)
source_bytes = file_obj.revisions[-1].read_bytes()

fillet = bracket_step.parse_fillet_mm(source_bytes)
print(f"Downloaded {len(source_bytes)} bytes")
print(f"Design parameter — fillet radius: {fillet:.1f} mm")

## 4. Do the work (outside Istari)

This is the part Istari never sees: our local verification battery. It runs ten checks and writes a result file for each, plus a JUnit XML summary. The **stress** check is physics-driven — a smaller fillet concentrates stress, so the R3 design pushes peak stress above the allowable and **fails**.

We render the results as a table so the verdict is obvious at a glance.

In [ ]:
import pandas as pd
from pathlib import Path
from run_workflow import run_battery, write_junit

work = Path("_notebook_run")
src = work / "bracket.step"
if not src.exists():
    raise FileNotFoundError(f"{src.resolve()} missing — run §2 first")
source_bytes = src.read_bytes()

results = run_battery(source_bytes, work / "artifacts")

def results_frame(results):
    df = pd.DataFrame([{"check": r.name,
                        "result": "PASS" if r.passed else "FAIL",
                        "detail": r.summary} for r in results])
    return df.style.map(
        lambda v: "color:#15803d;font-weight:600" if v == "PASS"
        else ("color:#dc2626;font-weight:600" if v == "FAIL" else ""),
        subset=["result"])

results_frame(results)

A couple of the outputs, shown inline — a generated thermal map and the stress check that failed:

In [ ]:
from IPython.display import Image, display

art = work / "artifacts"
display(Image(filename=str(art / "04_thermal_map.png")))
print(json.dumps(json.loads((art / "05_stress_analysis.json").read_text()), indent=2))

In [ ]:
overall = "SUCCESS" if all(r.passed for r in results) else "FAILED"
junit = write_junit(results, art / "00_verification_results.xml", src.name)
print("Overall verdict:", overall)

## 5. Upload the outputs to Istari

Now we push each result file up as a **workflow output**. Each call uploads the file and returns an ID we'll attach to the log entry. Workflow outputs live on the system but stay separate from the tracked configuration files — they won't show up in snapshots.

In [ ]:
upload_paths = [junit] + [r.artifact_path for r in results]
output_ids = []
for p in upload_paths:
    wo = v3.create_workflow_output(system_id=SYSTEM_ID, path=p)
    output_ids.append(wo.id)
    print(f"  uploaded {p.name}")
print(f"\n{len(output_ids)} workflow outputs uploaded")

## 6. Record the workflow log entry

A single call ties it all together: a title, the **status** (`FAILED` here), the **configuration** the run targeted, and the output IDs. This is the one call that creates a durable, linked record on the system.

In [ ]:
entry = v3.create_workflow_log_entry(
    system_id=SYSTEM_ID,
    workflow_log_entry_create_dto=WorkflowLogEntryCreateDto(
        title="Verification battery — iter-1 (initial design)",
        status=overall,                # SUCCESS | FAILED | UNSPECIFIED
        configuration_id=CONFIG_ID,
        workflow_output_ids=output_ids,
    ),
)
print("Created entry:", entry.id, "->", entry.status)
print("Open the Workflow log tab:", f"{UI_URL}/systems/{SYSTEM_ID}")

> **Pause here on the call.** Open the system link, go to the **Workflow log** tab, and click the failed entry. You can see the metadata, the configuration it points at, and preview or download each of the 11 output files.

## 7. Close the loop: revise the design

The engineer reacts to the failure by enlarging the fillet from R3 to **R5** to relieve the stress concentration, and commits that as a **new revision of the same model**. In Istari this is a normal file update — the system now has two revisions of the bracket.

(In the live UI demo this upload happens by drag-and-drop; here we do it via the SDK so the notebook runs top-to-bottom.)

In [ ]:
src.write_text(bracket_step.good())         # redesign: R5 fillet
client.update_model(model_id=MODEL_ID, path=src,
                    description="Bracket — R5 redesign", version_name="v2-R5-fillet")

# Capture a new snapshot and advance the baseline so the system shows the R5 revision.
istari_helpers.capture_and_show(client, SYSTEM_ID, CONFIG_ID)
print("Committed new revision with fillet:",
      f"{bracket_step.parse_fillet_mm(src.read_bytes()):.1f} mm")

## 8. Re-run the workflow against the new revision

Same workflow, no code changes. It pulls the **latest** revision (now R5), and this time every check passes — including stress, because the larger fillet brings peak stress back under the allowable.

In [ ]:
source_bytes = client.get_file(file_id=FILE_ID).revisions[-1].read_bytes()
print("Now testing fillet:", f"{bracket_step.parse_fillet_mm(source_bytes):.1f} mm")

results = run_battery(source_bytes, work / "artifacts2")
results_frame(results)

In [ ]:
art2 = work / "artifacts2"
overall = "SUCCESS" if all(r.passed for r in results) else "FAILED"
junit2 = write_junit(results, art2 / "00_verification_results.xml", src.name)

upload_paths = [junit2] + [r.artifact_path for r in results]
output_ids = [v3.create_workflow_output(system_id=SYSTEM_ID, path=p).id for p in upload_paths]

entry2 = v3.create_workflow_log_entry(
    system_id=SYSTEM_ID,
    workflow_log_entry_create_dto=WorkflowLogEntryCreateDto(
        title="Verification battery — iter-2 (R5 fillet redesign)",
        status=overall,
        configuration_id=CONFIG_ID,
        workflow_output_ids=output_ids,
    ),
)
print("Created entry:", entry2.id, "->", entry2.status)

## 9. Write the verified results back onto the design

Because the run passed, we attach the verification report to the bracket model itself as an artifact named `verified-test-results-for-bracket.step.xml`. Now the design carries the evidence that it passed — anyone opening the model can see it.

In [ ]:
import shutil
verified = work / "verified-test-results-for-bracket.step.xml"
shutil.copyfile(junit2, verified)
artifact = client.add_artifact(model_id=MODEL_ID, path=verified,
                               display_name=verified.name,
                               description=f"Verified results (entry {entry2.id})")
print("Attached artifact:", artifact.id)

## 10. Scenario A recap — the design loop

On a single system, with no manual bookkeeping, we now have:

- **Two workflow log entries** — a `FAILED` run and a `SUCCESS` run — each linked to the configuration it was tested against.
- **All result files** from both runs, previewable and downloadable in the UI.
- **A verified test report** attached to the design that passed.

That's the external design loop — *fail → revise → pass → evidence on the record* — captured automatically from an external workflow.

# Scenario B — Tradespace exploration

The design loop above was one engineer iterating a single design. A second, very common use of external workflows is a **tradespace sweep**: trying many parameter combinations to find one that satisfies every requirement at once.

Here the "workflow" is a **pytest suite** that encodes the requirements (stress, mass, deflection). The engineer sweeps two design parameters — **fillet radius** and **wall thickness** — and runs the *same* workflow for each candidate, logging every attempt under one **campaign title**. Runs that don't satisfy the requirements are recorded `FAILED`; the first one that passes is recorded `SUCCESS`.

This highlights two things about the workflow log:

- it's fine to record **many runs with the same name** — each entry is an independent record, and
- **status** plus the **attached parameter set** are how you find *which* design point won.

> On the call, this is the moment to make the point: the workflow log isn't just for one-off runs — it's a durable record of a whole **campaign**.

### The requirement suite

This is a real `pytest` file. Each candidate's parameters are passed in as environment variables, and the suite records them into the JUnit report via `record_property`, so the test artifact itself carries the design point it was run against.

In [ ]:
print(Path("tradespace_tests.py").read_text())

### Run the sweep

For each candidate we (1) write its parameter set as a first-class output, (2) run the pytest suite to produce a real JUnit report, (3) upload both as workflow outputs, and (4) log the attempt under the shared campaign title. We stop at the first candidate that passes.

In [ ]:
import subprocess, sys, math

campaign = "Tradespace sweep — bracket fillet/thickness study"
grid = [(2, 3), (2, 5), (3, 3), (8, 2), (3, 6), (4, 6), (10, 2), (4, 3), (5, 3), (5, 4)]
ts_dir = work / "tradespace"; ts_dir.mkdir(exist_ok=True)

def evaluate(r, t):
    peak = 1500.0 / (math.sqrt(r) * t)
    return {"peak_stress_MPa": round(peak, 1), "safety_factor": round(276.0 / peak, 2),
            "mass_kg": round(0.30 + 0.10 * t + 0.03 * r, 2), "stiffness_index": round(r * t, 1)}

sweep_rows, winner = [], None
for attempt, (r, t) in enumerate(grid, 1):
    run_dir = ts_dir / f"cand_{attempt:02d}"; run_dir.mkdir(exist_ok=True)

    # 1. the candidate parameter set — recorded as a first-class output on the entry
    metrics = evaluate(r, t)
    params = {"attempt": attempt, "campaign": campaign,
              "parameters": {"fillet_radius_mm": r, "wall_thickness_mm": t}, "metrics": metrics}
    params_path = run_dir / "00_parameters.json"
    params_path.write_text(json.dumps(params, indent=2))

    # 2. run the pytest requirement suite for this candidate -> real JUnit report
    junit = run_dir / "pytest_results.xml"
    proc = subprocess.run(
        [sys.executable, "-m", "pytest", "tradespace_tests.py", "-q", f"--junitxml={junit}"],
        env={**os.environ, "TS_FILLET_MM": str(r), "TS_THICKNESS_MM": str(t)},
        capture_output=True, text=True)
    passed = proc.returncode == 0
    status = "SUCCESS" if passed else "FAILED"

    # 3. upload outputs; the parameters file carries the design point on the entry
    summary = f"fillet={r} mm, thickness={t} mm"
    param_id = v3.create_workflow_output(system_id=SYSTEM_ID, path=params_path,
                   display_name=f"parameters ({summary})",
                   description=f"Candidate {attempt}: {summary}").id
    junit_id = v3.create_workflow_output(system_id=SYSTEM_ID, path=junit).id

    # 4. log this attempt under the SAME campaign title
    v3.create_workflow_log_entry(system_id=SYSTEM_ID,
        workflow_log_entry_create_dto=WorkflowLogEntryCreateDto(
            title=campaign, status=status, configuration_id=CONFIG_ID,
            workflow_output_ids=[param_id, junit_id]))

    sweep_rows.append({"attempt": attempt, "fillet_mm": r, "thickness_mm": t,
                       "SF": metrics["safety_factor"], "mass_kg": metrics["mass_kg"],
                       "stiffness": metrics["stiffness_index"], "result": status})
    print(f"  attempt {attempt:2d}: fillet={r:>2} thickness={t} -> {status}")
    if passed:
        winner = params["parameters"]; break

print("\nWinning configuration:", winner)

The sweep at a glance — the metrics that drove each verdict, and the candidate that finally satisfied all three requirements:

In [ ]:
def color_status(v):
    return ("color:#15803d;font-weight:600" if v == "SUCCESS"
            else "color:#dc2626;font-weight:600" if v == "FAILED" else "")

pd.DataFrame(sweep_rows).style.map(color_status, subset=["result"])

### The campaign in the workflow log

Every attempt is its own entry sharing the campaign title. We can pull the whole campaign back with a title filter and see — at a glance, by `status` — which design point won. Click into the `SUCCESS` entry in the UI and its attached `parameters` output records exactly which fillet and thickness passed.

In [ ]:
page = v3.list_workflow_log_entries(system_id=SYSTEM_ID, title=[campaign], size=100)
passed_n = sum(e.status == "SUCCESS" for e in page.items)
print(f"{len(page.items)} entries share the campaign title — {passed_n} SUCCESS, {len(page.items) - passed_n} FAILED")

pd.DataFrame([{"status": e.status, "files": e.file_count,
               "created": e.created.strftime("%H:%M:%S"), "entry": str(e.id)[:8]}
              for e in page.items]).style.map(color_status, subset=["status"])

# Wrap-up

Across two scenarios, on systems in Istari, we captured external work as a durable record with **zero manual bookkeeping**:

- **Scenario A — design loop:** a failing run, the revision that fixed it, the passing run, and the verified report written back onto the design.
- **Scenario B — tradespace sweep:** ten same-named runs of a pytest requirement suite, each carrying its parameter set, with `status` pinpointing the winning design point.

In both cases the workflow log ties every run to the configuration it was tested against — so months later you can still answer *what was run, against which design, and how it turned out.*

In [ ]:
print("Review both scenarios in the Workflow log tab:")
print(f"  {UI_URL}/systems/{SYSTEM_ID}")

### Learn more

- **SDK guide:** External workflow logs — `create_workflow_output`, `create_workflow_log_entry`, listing and reading entries.
- **User guide:** the **Workflow log** tab on a system (experimental — enable under **Application Settings → Experimental — External Workflows**).

## Teardown (optional)

When you're done — or before practicing the run again — archive the demo system so
repeated runs don't pile up on the instance. This is safe to skip during the live
call; run it afterwards. Archiving is reversible (the system can be restored).

In [ ]:
client.archive_system(system_id=SYSTEM_ID)
print("Archived system", SYSTEM_ID)